In [ ]:
# 全局设置
import datetime as dt
import warnings
from pandas.errors import PerformanceWarning
warnings.filterwarnings('ignore', category=PerformanceWarning)
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams['font.sans-serif'] = ['SimHei']
matplotlib.rcParams['axes.unicode_minus'] = False# 正确显示负号
from IPython.display import HTML

from QuantStudio import __QS_MainPath__

# 业绩归因

业绩归因用于将投资组合的超额收益分解为不同来源的贡献，是评估投资经理能力来源的核心工具。QuantStudio 的业绩归因功能位于 `BackTest.PerformanceAnalysis` 模块，建立在回测基本框架之上。

> **前置阅读**：回测框架的整体架构、`BTNode`/`BTReport` 基类、执行流程（三层嵌套 `Cache → Context → Engine`）以及"算子-节点分离"设计模式，请先参阅 **[基本框架](基本框架.ipynb)**。计算图与引擎的基础知识请参阅 **[计算图框架](../Core/计算图框架.ipynb)** 和 **[计算引擎](../Core/计算引擎.ipynb)**。

## 模块概览

`QuantStudio.BackTest.PerformanceAnalysis` 提供以下归因模型：

| 模型 | 算子（计算层） | 回测节点（报告层） | 功能 |
|------|---------------|-------------------|------|
| Brinson 模型 | `CalcBrinsonModel` | `BrinsonModel` | 将超额收益分解为资产配置、个股选择和交互作用 |

每个模型遵循"算子 + 回测节点"两层架构：算子（`PanelOperator` 子类）封装纯计算逻辑，回测节点（`BTNode` 子类）负责统计汇总和 HTML 报告生成。

---

## Brinson 模型

Brinson 和 Falcher（1985 年）最早对业绩归因进行研究，由他们所创建的 Brinson 模型将业绩归因为四个部分：资产配置、个股选择、交互作用和基准收益。

### 单期 Brinson 模型

当前时刻为 $t$，考虑时段 $[t-1, t]$ 上的业绩归因，假设在 $t-1$ 构建了投资组合，且在 $[t-1, t]$ 之间没有交易。

记：
* 业绩基准组合里个股 $j$ 的权重为 $w^b_j$，资产 $i$ 的权重为 $W^b_i=\sum\limits_{j\in i}w^b_j$；
* 实际投资组合里个股 $j$ 的权重为 $w^p_j$，资产 $i$ 的权重为 $W^p_i=\sum\limits_{j\in i}w^p_j$；
* 个股 $j$ 的收益率为 $r_j$。

考虑四个投资组合：

* **P1 业绩基准组合**：其中资产 $i$ 的收益率为：
$$
R^b_i=\frac{\sum\limits_{j\in i}w^b_jr_j}{W^b_i}
$$
组合收益率：
$$
R_{P1} = R_b = \sum\limits_iW^b_iR^b_i=\sum\limits_i\sum\limits_{j\in i}w^b_jr_j
$$

* **P2 主动资产配置组合**：自主选择资产配置的比例，但是每个资产类别内部则完全按照其业绩基准配置，即每个资产 $i$ 的收益等于在基准中资产 $i$ 的收益 $R^b_i$，而每个资产 $i$ 的权重等于在实际组合中资产 $i$ 的权重 $W^p_i$，则其组合收益率：
$$
R_{P2} = \sum\limits_iW^p_iR^b_i=\sum\limits_i\frac{W^p_i}{W^b_i}\sum\limits_{j\in i}w^b_jr_j
$$

* **P3 主动股票选择组合**：完全按照业绩基准进行资产类别的配置，但是每个资产内部能够自主选择个股，即组合中每个资产 $i$ 的权重等于基准中资产 $i$ 的权重 $W^b_i$，每个资产 $i$ 的收益等于在实际组合中资产 $i$ 的收益 $R^p_i$，则其组合收益率：
$$
R_{P3} = \sum\limits_iW^b_iR^p_i=\sum\limits_i\frac{W^b_i}{W^p_i}\sum\limits_{j\in i}w^p_jr_j
$$

* **P4 实际投资组合**：其中资产 $i$ 的收益率为：
$$
R^p_i=\frac{\sum\limits_{j\in i}w^p_jr_j}{W^p_i}
$$
组合收益率：
$$
R_{P4} = R_p = \sum\limits_iW^p_iR^p_i=\sum\limits_i\sum\limits_{j\in i}w^p_jr_j
$$

Brinson 模型将超额收益分解为：

**1. 资产配置收益（Return of Asset Allocation）**

假设我们能够自主选择决定组合中资产配置的比例，但是在每一个资产类别内部则完全按照该基准组合配置，那么该组合的收益率超过基准收益率的部分称为资产配置收益：
$$
R_{AA} = R_{P2} - R_{P1} = \sum\limits_i(W^p_i-W^b_i)R^b_i
$$

**2. 个股选择收益（Return of Stock Selection）**

假设我们完全按照基准的比例进行资产类别配置，但是在每一个资产类别内部则能够自主进行个股选择，那么该组合的收益率超过基准收益率的部分称为个股选择收益：
$$
R_{SS} = R_{P3} - R_{P1} = \sum\limits_iW^b_i(R^p_i-R^b_i)
$$

**3. 交互作用收益（Interaction）**

投资组合的超额收益不仅来自资产配置收益和个股选择收益，还有一部分是由于二者的交互作用所带来的收益：
$$
R_{IN} = R_{P4} - R_{P3} - R_{P2} + R_{P1} = \sum\limits_i(W^p_i-W^b_i)(R^p_i-R^b_i)
$$

则总超额收益为：
$$
R_{p-b} = R_{P4} - R_{P1} = R_{AA} + R_{SS} + R_{IN}
$$

**修正的资产配置收益**

Damien Laker 提出原资产配置收益的定义可能会带来错误，故将之修改为：
$$
R'_{AA} = \sum\limits_i(W^p_i-W^b_i)(R^b_i-R_b)
$$

尽管新定义加入了基准收益，但由于权重之和都为 1，所以 $R'_{AA}=R_{AA}$。两者的差异在于：原定义强调超额资产配置的收益来源于对收益更高的资产 $i$ 的超配，或对收益低的资产 $j$ 的低配；而新定义强调对表现优于基准总收益的类别 $i$ 超配，或劣于基准总收益的类别 $j$ 低配，后者更符合实际。

### 多期 Brinson 模型

两个计算时点之间没有调整策略持仓，组合多期总收益是单期收益按照复利计算的结果：
$$
R_p = (1+R_{p,1})(1+R_{p,2})\cdots(1+R_{p,T}) - 1
$$

基准总收益为：
$$
R_b = (1+R_{b,1})(1+R_{b,2})\cdots(1+R_{b,T}) - 1
$$

由于复合收益率无法分解为单期收益率的和，考虑对数收益率：
$$
\operatorname{log}(1+R_p) - \operatorname{log}(1+R_b) = \sum\limits_{t=1}^{T}\left[\operatorname{log}(1+R_{p,t}) - \operatorname{log}(1+R_{b,t})\right]
$$

对于单期有分解：
$$
\operatorname{log}(1+R_{p,t}) - \operatorname{log}(1+R_{b,t}) = k_t(R_{p,t} - R_{b,t}) = k_t(R_{AA,t} + R_{SS,t} + R_{IN,t})
$$

其中：
$$
k_t = \frac{\operatorname{log}(1+R_{p,t}) - \operatorname{log}(1+R_{b,t})}{R_{p,t} - R_{b,t}}
$$

从而多期总超额收益有如下分解：
$$
\begin{align}
    R_p - R_b &= R_{AA} + R_{SS} + R_{IN} \\
    R_{AA} &= \sum\limits_{t=1}^{T}\frac{k_t}{k}R_{AA,t} \\
    R_{SS} &= \sum\limits_{t=1}^{T}\frac{k_t}{k}R_{SS,t} \\
    R_{IN} &= \sum\limits_{t=1}^{T}\frac{k_t}{k}R_{IN,t}
\end{align}
$$

其中：
$$
k = \frac{\operatorname{log}(1+R_p) - \operatorname{log}(1+R_b)}{R_p - R_b}
$$

### CalcBrinsonModel — 计算算子

`CalcBrinsonModel` 是一个 `PanelOperator`（面板算子），对每个时点计算投资组合相对于基准的 Brinson 业绩归因分解。

#### 构造参数

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `section_ids` | `List[str]` | — | 绩效分析因子的截面 ID 序列，如使用行业分类则该截面应为所有行业列表 |
| `descriptor_ids` | `List[str]` | — | 依赖因子的截面 ID 序列 |
| `lookback` | `int` | `31` | 在时间标尺上的回溯期数 |

**前提条件**：
* 投资组合的权重之和为 1，与 1 的差值部分归为现金
* 两个计算时点之间没有调整策略持仓

#### `__call__` 参数

| 参数 | 类型 | 说明 |
|------|------|------|
| `p` | `Factor` | 待分析的投资组合因子 |
| `price` | `Factor` | 证券价格或净值因子，用于计算收益率 |
| `cat_data` | `Factor` | 类别因子（如行业），作为资产分类依据 |
| `bmk` | `Optional[Factor]` | 基准投资组合因子，为 `None` 时表示没有基准，考察绝对收益 |
| `factor_args` | `dict` | 传递给 Brinson 因子的参数（如 `CalcDTRuler` 指定计算时点） |

**返回值**：复合类型因子（`CompoundType`），每行包含 8 个字段：

| 字段 | 类型 | 说明 |
|------|------|------|
| `BMK` | `double` | 业绩基准组合在各大类资产上的权重 |
| `BMKR` | `double` | 业绩基准组合在各大类资产上的收益率 |
| `TP` | `double` | 目标投资组合在各大类资产上的权重 |
| `TPR` | `double` | 目标投资组合在各大类资产上的收益率 |
| `AA` | `double` | 资产配置超额收益（Return of Asset Allocation） |
| `SS` | `double` | 个券选择超额收益（Return of Stock Selection） |
| `IN` | `double` | 交互作用（Interaction） |
| `AAA` | `double` | 修正的资产配置收益 |

### BrinsonModel — 回测报告节点

`BrinsonModel` 是继承自 `BTNode` 的回测节点，接收 `CalcBrinsonModel` 产生的因子作为依赖，汇总统计并生成 HTML 报告。

#### 参数（`__QS_ArgClass__`）

| 参数 | 类型 | 默认值 | 说明 |
|------|------|--------|------|
| `Name` | `str` | `"Brinson 绩效分析模型"` | 节点名称（冻结） |

#### 构造方法

```python
BrinsonModel(
    brinson: Factor,           # CalcBrinsonModel 产生的 Brinson 因子
    args: dict = {},           # 参数集（如 {"GenReport": True}）
    config_file: Optional[str] = None,
    **kwargs
)
```

#### 输出数据结构（`backward_compute` 返回的字典）

**单期数据**（按资产类别分列，行为时点索引）：

| 键 | 类型 | 说明 |
|------|------|------|
| `"策略组合资产权重"` | `DataFrame` | 各期各资产在策略组合中的权重 |
| `"基准组合资产权重"` | `DataFrame` | 各期各资产在基准组合中的权重 |
| `"策略组合资产收益"` | `DataFrame` | 各期各资产的策略组合收益率 |
| `"基准组合资产收益"` | `DataFrame` | 各期各资产的基准组合收益率 |
| `"主动资产配置超额收益"` | `DataFrame` | 各期各资产的资产配置超额收益 |
| `"主动个券选择超额收益"` | `DataFrame` | 各期各资产的个券选择超额收益 |
| `"主动资产配置组合收益"` | `DataFrame` | 资产配置组合总收益（= 资产配置超额收益 + 基准资产收益） |
| `"主动个券选择组合收益"` | `DataFrame` | 个券选择组合总收益（= 个券选择超额收益 + 基准资产收益） |
| `"交互作用超额收益"` | `DataFrame` | 各期各资产的交互作用收益 |
| `"总超额收益"` | `DataFrame` | 各期各资产的总超额收益（= 策略收益 - 基准收益） |
| `"主动资产配置组合收益(修正)"` | `DataFrame` | 修正的资产配置组合收益（Damien Laker 定义） |

**汇总数据**：

| 键 | 类型 | 说明 |
|------|------|------|
| `"总计"` | `DataFrame` | 各期全资产的汇总数据（策略/基准/配置/选择/交互等各项收益的时序求和） |
| `"多期综合"` | `DataFrame` | 多期复合收益及分解，含"总计"行的调整后多期归因结果 |

报告 HTML 展示一张汇总表格，包含各资产类别的多期综合收益分解（策略组合收益、基准组合收益、主动资产配置、主动个券选择、交互作用和总超额收益）。

#### 构造示例

```python
from QuantStudio.BackTest.PerformanceAnalysis.BrinsonModel import CalcBrinsonModel, BrinsonModel

# 创建 Brinson 算子并作用于因子
BrinsonFactor = CalcBrinsonModel(
    section_ids=IndustryList,       # 资产类别列表（如行业）
    descriptor_ids=SectionIDs       # 截面 ID 序列
)(
    Portfolio,                      # 策略组合权重因子
    price=Price,                    # 价格因子
    cat_data=Industry,              # 类别因子（行业）
    bmk=BmkPortfolio,               # 基准组合权重因子
    factor_args={"CalcDTRuler": BalanceDTs}
)

# 创建 Brinson 回测节点
BrinsonNode = BrinsonModel(BrinsonFactor, args={"GenReport": True})
```

### 完整示例

下面展示一个完整的 Brinson 业绩归因流程，使用 EP（TTM）因子构建策略组合和基准组合，按行业分类进行归因分析。

> **执行流程**（三层嵌套 `Cache → Context → Engine`）以及 `BTReport` 的报告汇总机制已在 **[基本框架](基本框架.ipynb)** 中详细说明，此处不再赘述。

In [ ]:
# 参数设置
from QuantStudio.Factor.HDF5DB import HDF5DB
FDB = HDF5DB(args={"MainDir": Path(__QS_MainPath__).parent / "docs/data/HDF5"}).connect()

StartDT, EndDT = dt.datetime(2025, 1, 1), dt.datetime(2025, 3, 31)# 数据起止时间
TestStartDT, TestEndDT = dt.datetime(2025, 2, 28), EndDT# 测试起止时间

FT = FDB.getTable("stock_cn_day_bar")
DTRuler = FT.getDateTime(start_dt=StartDT, end_dt=EndDT)
TestDTs = FT.getDateTime(start_dt=TestStartDT, end_dt=TestEndDT)
SectionIDs = IDs = FT.getID()

# 再平衡时点序列
from QuantStudio.Tools.DateTimeFun import getMonthLastDateTime
BalanceDTs = getMonthLastDateTime(DTRuler)# 月末

In [ ]:
# 导入回测相关模块
from QuantStudio.Core.CalcEngine import Engine
from QuantStudio.Core.Node import DTLocalContext, DTInitData
from QuantStudio.Factor.Factor import FactorContext
from QuantStudio.Factor.FactorCache import FeatherFactorCache
from QuantStudio.Factor.BasicOperator import rename
import QuantStudio.Factor.FactorOperator as fo
from QuantStudio.BackTest.BackTestModel import BTReport
from QuantStudio.BackTest.PerformanceAnalysis.BrinsonModel import CalcBrinsonModel, BrinsonModel

# 获取数据因子
FT = FDB.getTable("stock_cn_status")
Mask = (FT.getFactor("if_listed")==1)

FT = FDB.getTable("stock_cn_day_bar")
Price = FT.getFactor("close")

FT = FDB.getTable("stock_cn_industry")
Industry = FT.getFactor("industry")
IndustryList = sorted(pd.unique(Industry.readData(ids=None, dts=None).values.flatten()))

FT = FDB.getTable("stock_cn_factor_value")
EP = FT.getFactor("ep_ttm")

# 构建策略和基准组合
EPRank = fo.SectionRank(ascending=True, uniformization=True)(EP, mask=Mask, factor_args={"CalcDTRuler": BalanceDTs})

# 策略组合: EP TTM 属于前 20% 的股票等权配置
Portfolio = (EPRank >= 0.8)
Portfolio = rename(Portfolio / fo.Aggregate(aggr_func=np.nansum)(Portfolio), factor_name="Portfolio")# 归一化

# 基准组合: EP TTM 属于前 50% 的股票等权配置
BmkPortfolio = (EPRank >= 0.5)
BmkPortfolio = rename(BmkPortfolio / fo.Aggregate(aggr_func=np.nansum)(BmkPortfolio), factor_name="BmkPortfolio")# 归一化

# ========== 构建 Brinson 归因 ==========
BrinsonFactor = CalcBrinsonModel(
    section_ids=IndustryList,
    descriptor_ids=SectionIDs
)(
    Portfolio,
    price=Price,
    cat_data=Industry,
    bmk=BmkPortfolio,
    factor_args={"CalcDTRuler": BalanceDTs}
)
BrinsonNode = BrinsonModel(BrinsonFactor, args={"GenReport": True})

# ========== 创建报告并执行 ==========
NodeList = [BrinsonNode]
Report = BTReport(bt_node_list=NodeList)

CacheDir = Path(__QS_MainPath__).parent / "docs/data/Cache"
if not CacheDir.exists(): CacheDir.mkdir(parents=True)

with FeatherFactorCache(args={"DTRuler": DTRuler, "CacheDir": CacheDir, "StartMode": "new"}) as Cache:
    with FactorContext(DTRuler=DTRuler, SectionIDs=SectionIDs, DataCache=Cache) as Context:
        with Engine() as ExecEngine:
            Rslt = ExecEngine.run(
                [Report], Context,
                fwd_data_list=[DTLocalContext(DTs=TestDTs)],
                init_data_list=[DTInitData(DTRange=(TestDTs[0], TestDTs[-1]))]
            )

display(HTML(Rslt[0]["Report"]))

### 输出结构

报告 HTML 展示各资产类别的多期综合收益分解表格，包含策略组合收益、基准组合收益、主动资产配置超额收益、主动个券选择超额收益、交互作用超额收益和总超额收益。

`BrinsonModel` 的 `backward_compute` 返回字典的各键可通过编程方式访问各维度的归因数据，例如：

```python
# 访问归因节点输出
brinson_output = Rslt[0]

# 单期各类资产归因数据
aa_asset = brinson_output["主动资产配置超额收益"]    # 各期各行业资产配置收益
ss_asset = brinson_output["主动个券选择超额收益"]    # 各期各行业个券选择收益
in_asset = brinson_output["交互作用超额收益"]        # 各期各行业交互作用收益

# 各期汇总（全资产加总）
total = brinson_output["总计"]                       # 各期各项收益汇总
total["主动资产配置超额收益"]                        # 各期资产配置总超额收益时序

# 多期复合归因结果
multi_period = brinson_output["多期综合"]            # 多期综合收益分解
multi_period.loc["总计"]                              # 调整后的多期归因总计
```